# 🎯 [MOV01_M07] 우회로 수송 명령 하달을 위한 작전군수관 HITL 통제 워크플로우 설계 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV01/mov01_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 3) |
| **미션 ID** | MOV01_M07 |
| **미션 주제** | 우회로 수송 명령 하달을 위한 작전군수관 HITL 통제 워크플로우 설계 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수송 소요를 데이터로 예측해 군수 수송계획을 세우는 일은 구간별 통과 용량과 병목을 함께 보아야 정확해진다. 본 프로젝트는 수송 소요예측에서 예측 대상 정의, 시계열·피처 구성, 기준모델 비교, 설명가능성, 업무 임계값, 실적 대조를 익히도록 교육용 가상 노선망 자료를 코드로 만들어 구간별 수송 소요와 지체를 예측한다. 학습자는 병목 데이터 정의, 지체 전처리, BPR 기준 모델, 최대유량 기반 병목 식별과 피처 모델, 우회 소요·유류 산출과 앙상블 모델, 혼잡도 대시보드와 요인 설명, 작전군수관 승인(HITL)과 선형계획 최적화, 3개 구간 동시 차단 스트레스 시험, AAR 재보정과 ROI, 재학습 파이프라인 통합을 차례로 수행한다. 최종 목표는 구간별 소요·지체를 예측해 병목을 사전에 찾고 우회 물량 배분안을 만들되, 우회 명령의 하달 여부는 반드시 사람이 결정하는 수송 소요예측 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(작전군수처, 수송대대)에 즉시 도입 가능한 "완성형 국방 AI 수송 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 기술 명세 및 운영 매뉴얼** | MSR 통과 능력 데이터 명세서(`docs/mov01_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mov01_operations_guide.md`) | 미션 1, 10 |
| **`src/`** | **군수 데이터 정제 및 분석 모듈** | 통과 지체시간 계산 전처리 파이프라인(`src/data_preprocessing.py`), BPR 기반 1차 지체 예측 Baseline(`src/baseline_delay.py`) | 미션 2, 3 |
| **`notebooks/`** | **통계 검증 및 심층 EDA** | MSR 구간 지체 탐색적 데이터 분석 및 파레토 분석(`notebooks/mov01_eda_analysis.ipynb`) | 미션 3 |
| **`src/`** | **AI 예측 & 우회로 탐색 엔진** | Max-Flow 기반 병목 사전 식별 및 우회로 탐색(`src/network_flow.py`), 유류 소모 및 소요시간 산출(`src/route_cost_engine.py`) | 미션 4, 5 |
| **`src/`** | **작전 상황판 & 스트레스 모의** | Streamlit 기반 전술 COP 대시보드(`src/dashboard_app.py`), 3개 축선 봉쇄 스트레스 시험(`src/stress_test.py`) | 미션 6, 8 |
| **`pipeline/`** | **HITL 통제 & 지속적 학습** | 작전군수관 3단계 승인 파이프라인(`pipeline/hitl_workflow.py`), 챔피언/챌린저 모델 자동 재학습 | 미션 7, 10 |
| **`reports/`** | **검증 및 AAR 리포트** | 3개 축선 봉쇄 스트레스 검증 보고서(`reports/mov01_stress_test_report.md`), AAR 오차 분석 및 ROI 보고서(`reports/mov01_roi_analysis.md`) | 미션 8, 9 |
| **`root`** | **통합 배포 패키지** | 통합 실행 진입점(`main.py`), 국방 망분리 Dockerfile 패키지 | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 우회로 수송 명령 하달을 위한 작전군수관 HITL 통제 워크플로우 설계
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

우회로 수송 명령 하달을 위한 작전군수관 HITL 워크플로우를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **우회 수송 명령의 하달 여부를 작전군수관이 직접 승인·반려하는 인간 개입(HITL) 결재 워크플로우를 설계하고, 선형계획으로 우회 물량 배분을 최적화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **선형계획(LP)으로 구간 용량 상한 아래에서 경로별 물량을 배분하고 시나리오 6종과 과거 30일 조건에서 고정 배분 대비 비용 절감률을 백테스트하며 돌발 수요 민감도 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/optimization/mov01_simulator.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현실적 군수 제약조건 반영도 및 시나리오별 대안 제시 적절성

#### 미션 PASS 조건

- 정량 기준: 최적화 적용 시 비용 15% 절감 또는 재고 회전율 20% 향상


- 진행 기준: 선형/정수 계획법(LP/MIP) 최적화 해 도출 및 시뮬레이션 완료


- 검증 방법: 과거 실적 데이터 대비 최적화 결과 비교 백테스팅


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""선형계획(LP)으로 우회 수송 물량 배분을 최적화해 과거 관행 대비 비용 절감을 백테스트하고, 우회 수송 명령을 작전군수관이 직접 승인·반려하는 HITL 결재 워크플로우(권한·2단계 결재·해시 체인 감사 로그)를 구현한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 노선망·구간 통과량 생성기 (실제 도로·지명·부대 없음)
# 가상 출발지(SRC)에서 가상 목적지(DST)까지 3개 축선(A·B·C)과 연결로 2개, 구간 10개. 150일치 피크시간 통과량·지체 시간을 만든다.
# ---------------------------------------------------------------------------
SEGS = pd.DataFrame([
    ("A1", "SRC", "J1", 12, 130, "일반"), ("A2", "J1", "J2", 8, 70, "교량"), ("A3", "J2", "DST", 10, 140, "교차로"),
    ("B1", "SRC", "J3", 15, 100, "일반"), ("B2", "J3", "J2", 9, 80, "터널"), ("B3", "J3", "DST", 14, 95, "일반"),
    ("C1", "SRC", "J4", 18, 70, "일반"), ("C2", "J4", "DST", 20, 75, "교량"),
    ("X1", "J1", "J3", 7, 50, "교차로"), ("X2", "J4", "J3", 10, 60, "일반")],
    columns=["seg", "u", "v", "km", "cap", "kind"])               # cap = 피크시간 통과 용량(대/시, 교육용 가정값)
KSPD = {"일반": 55, "교량": 40, "터널": 45, "교차로": 45}           # 구간 유형별 자유 주행 속도(km/h, 가정값)
SEGS["t0"] = SEGS["km"] / SEGS["kind"].map(KSPD) * 60              # 자유 주행 시간(분)
UTIL = dict(A1=.78, A2=.85, A3=.66, B1=.70, B2=.72, B3=.64, C1=.55, C2=.58, X1=.45, X2=.40)   # 평시 이용률


def bpr(t0, v, cap, a=0.15, b=4.0):
    """BPR 지체 함수(교통공학의 표준형): 통과량/용량 비가 커질수록 지체가 급증한다. (분)"""
    # [TODO : 학생 실습 7] 우회로 수송 명령 하달을 위한 작전군수관 HITL 통제 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_data(seed=11, days=150):
    """구간 x 일자 패널. v=실제 피크 통과량, plan_v=수송계획 수량(긴급 불출 미반영), delay=실제 지체(분)."""
    rng = np.random.default_rng(seed)
    d_idx = np.arange(days)
    lvl = np.array([1.0, 1.08, 1.04, 1.10, 1.02, 0.82, 0.78])[d_idx % 7] * (1 + 0.0012 * d_idx)
    lvl = lvl * np.where((d_idx >= 100) & (d_idx < 110), 1.30, 1.0)       # 수송량 폭증 기간(가상)
    prev, rows = {s: 0.0 for s in SEGS["seg"]}, []
    for d in d_idx:
        rain = bool(rng.random() < 0.18)
        for r in SEGS.itertuples():
            maint, urgent = bool(rng.random() < 0.06), bool(rng.random() < 0.05)   # 비계획 정비 / 긴급 불출
            v = r.cap * UTIL[r.seg] * lvl[d] * (1.25 if urgent else 1.0) * rng.normal(1, 0.05)
            cap_eff = r.cap * (0.65 if maint else 1.0) * (0.92 if rain else 1.0)
            delay = max(r.t0 * 0.2 * (v / cap_eff) ** 3.5 + 0.25 * prev[r.seg] + rng.normal(0, 0.4), 0.0)
            prev[r.seg] = delay
            rows.append(dict(day=int(d), dow=int(d % 7), seg=r.seg, kind=r.kind, km=r.km, cap=r.cap, t0=r.t0,
                             plan_v=v / (1.25 if urgent else 1.0) * rng.normal(1, 0.04), v=v, maint=int(maint), urgent=int(urgent),
                             rain=int(rain), cap_eff=cap_eff, delay=delay))
    df = pd.DataFrame(rows)
    df["transit"] = df["t0"] * (1 + 0.08 * df["rain"]) + df["delay"]
    return df


def mape(y, p):
    # [TODO : 학생 실습 7] 우회로 수송 명령 하달을 위한 작전군수관 HITL 통제 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 노선망 흐름 도구: 최대유량(Edmonds-Karp), 증분 배정(혼잡 반영 최단시간 경로), 경로 열거
# ---------------------------------------------------------------------------
CAP = dict(zip(SEGS["seg"], SEGS["cap"]))
T0 = dict(zip(SEGS["seg"], SEGS["t0"]))
KM = dict(zip(SEGS["seg"], SEGS["km"]))
ARCS = {r.seg: (r.u, r.v) for r in SEGS.itertuples()}
SRC, DST = "SRC", "DST"


def seg_time(s, flow, cap=None, a=0.2, b=3.5):
    """구간 통과시간(분) = 자유주행 + 혼잡 지체. cap을 주면 용량을 대체(정비·기상 반영)."""
    r = flow / (cap or CAP[s])
    return T0[s] * (1 + a * r ** b + 6.0 * max(0.0, r - 1.0))        # 용량 초과분은 대기열로 급증


def max_flow(closed=(), capmul=None):
    """최대유량(대/시)과 최소 절단 구간(병목). capmul = {구간: 용량 배율}."""
    cap = {s: (0 if s in closed else CAP[s] * (capmul or {}).get(s, 1.0)) for s in CAP}
    res = {}
    for s, (u, v) in ARCS.items():
        res[(u, v)] = res.get((u, v), 0) + cap[s]
        res.setdefault((v, u), 0)
    flow = 0.0
    while True:
        prev, q = {SRC: None}, [SRC]
        for u in q:
            for (a, b), c in res.items():
                if a == u and c > 1e-9 and b not in prev:
                    prev[b] = u; q.append(b)
        if DST not in prev:
            break
        path, x = [], DST
        while prev[x]:
            path.append((prev[x], x)); x = prev[x]
        f = min(res[e] for e in path)
        for a, b in path:
            res[(a, b)] -= f; res[(b, a)] += f
        flow += f
    reach = set(prev)
    cut = [s for s, (u, v) in ARCS.items() if u in reach and v not in reach and cap[s] > 0]
    return flow, cut


def all_paths():
    """SRC -> DST 단순 경로(구간 이름 튜플) 전체."""
    out = []

    def dfs(node, segs, seen):
        if node == DST:
            out.append(tuple(segs)); return
        for s, (u, v) in ARCS.items():
            if u == node and v not in seen:
                dfs(v, segs + [s], seen | {v})
    dfs(SRC, [], {SRC})
    return out


PATHS = all_paths()


def assign(demand, closed=(), capmul=None, steps=40):
    """증분 배정: 수요를 steps조각으로 나눠, 전체 통과시간(대수 x 시간) 증가가 가장 작은 경로에 차례로 얹는다(시스템 최적 근사)."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    flow, pf = {s: 0.0 for s in CAP}, {}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for _ in range(steps):
        chunk = demand / steps
        best = min(ok, key=lambda p: sum((flow[s] + chunk) * seg_time(s, flow[s] + chunk, cap[s]) - flow[s] * seg_time(s, flow[s], cap[s]) for s in p))
        for s in best:
            flow[s] += chunk
        pf[best] = pf.get(best, 0) + chunk
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    avg = sum(pf[p] * tt[p] for p in pf) / demand
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": avg, "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


FIXED_SHARE = {("A1", "A2", "A3"): 0.32, ("B1", "B3"): 0.40, ("C1", "C2"): 0.28}      # 과거 관행: 축선별 고정 배분(가상)


def fixed_assign(demand, closed=(), capmul=None):
    """과거 관행 배분: 축선별 고정 비율. 막힌 축선 물량은 열려 있는 축선에 비율대로 재분배."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    live = {p: w for p, w in FIXED_SHARE.items() if not any(s in closed for s in p)}
    if not live:
        return None
    tot = sum(live.values())
    flow = {s: 0.0 for s in CAP}
    pf = {p: demand * w / tot for p, w in live.items()}
    for p, f in pf.items():
        for s in p:
            flow[s] += f
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


def lp_assign(demand, closed=(), capmul=None):
    """선형계획(LP): 구간 용량 상한(이용률 rho) 아래에서 자유주행시간 합을 최소화하는 경로별 물량. 불가능하면 rho를 1.0까지 완화."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for rho in (0.85, 0.95, 1.0):
        r = linprog([sum(T0[s] for s in p) for p in ok], A_ub=[[1.0 if s in p else 0.0 for p in ok] for s in CAP],
                    b_ub=[rho * cap[s] for s in CAP], A_eq=[[1.0] * len(ok)], b_eq=[demand], bounds=(0, None), method="highs")
        if r.status == 0:
            break
    else:
        return assign(demand, closed, capmul)                # 용량 초과 수요: 증분 배정으로 대체(과부하 표시)
    flow = {s: sum(r.x[i] for i, p in enumerate(ok) if s in p) for s in CAP}
    pf = {p: float(r.x[i]) for i, p in enumerate(ok) if r.x[i] > 1e-6}
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}, "rho": rho}


def cost_of(res, tv=1000, fuel_km=450):
    """시간당 비용(원, 교육용 가정): 대당 시간비용 tv원/분 + 유류비 fuel_km원/km."""
    return sum(f * (res["path_min"][p] * tv + sum(KM[s] for s in p) * fuel_km) for p, f in res["paths"].items())
# ---------------------------------------------------------------------------

# ---------------- 1) 최적화 시뮬레이터: 과거 관행(축선 고정 배분) vs LP ----------------
SC = [("평시", 200, (), None), ("수송량 폭증(+30%)", 260, (), None), ("A2 교량 정비(용량 65%)", 200, (), {"A2": .65}), ("A2 교량 폐쇄", 200, ("A2",), None),
      ("A2 폐쇄 + 우천", 200, ("A2",), {k: .92 for k in CAP}), ("B1 정비 + 폭증", 250, (), {"B1": .65})]
rows, plans = [], {}
for name, D, cl, cm in SC:
    f, l = fixed_assign(D, cl, cm), lp_assign(D, cl, cm)
    cf, cl_ = cost_of(f), cost_of(l)
    rows.append((name, D, f["avg_min"], l["avg_min"], cf / 1e4, cl_ / 1e4, 1 - cl_ / cf))
    plans[name] = (D, cl, cm, l, cf, cl_)
res = pd.DataFrame(rows, columns=["시나리오", "수요", "고정 배분(분)", "LP(분)", "고정 비용(만원/시)", "LP 비용(만원/시)", "절감률"])
print("[1] 시나리오별 비용 비교(시간·유류 단가는 교육용 가정값)")
print(res.assign(절감률=res["절감률"].map("{:.1%}".format)).round(1).to_string(index=False))
avg = res["절감률"].mean()
print(f"    시나리오 6종 평균 절감 {avg:.1%}, 우회가 필요한 4개 시나리오 평균 {res['절감률'].iloc[2:].mean():.1%} (목표 15% 이상: {'달성' if avg >= .15 else '미달'}), 평시는 {res['절감률'].iloc[0]:.1%}에 그친다")

# ---------------- 2) 백테스팅: 과거 30일(day 120~149) 실적 조건으로 재현 ----------------
df = make_data()
rec = []
for day, g in df[df.day >= 120].groupby("day"):
    D = float(g[g.seg.isin(["A1", "B1", "C1"])]["plan_v"].sum())
    cm = {r.seg: r.cap_eff / r.cap for r in g.itertuples() if r.cap_eff < r.cap}
    f, l = fixed_assign(D, (), cm), lp_assign(D, (), cm)
    rec.append((day, D, cost_of(f), cost_of(l)))
bt = pd.DataFrame(rec, columns=["day", "D", "fix", "lp"])
bt["save"] = 1 - bt["lp"] / bt["fix"]
print(f"\n[2] 백테스팅(30일): 평균 절감 {bt['save'].mean():.1%}(목표 15% 이상: {'달성' if bt['save'].mean() >= .15 else '미달'}), 최대 {bt['save'].max():.1%}(day {int(bt.loc[bt['save'].idxmax(), 'day'])}), 절감 0 이하인 날 {int((bt['save'] <= 0).sum())}일")
print(f"    절감 10% 미만인 날 {int((bt['save'] < .10).sum())}일: 효과는 정비·폭증이 겹친 날에 집중된다. 과거 관행 배분은 가정이므로 실제 부대 실적으로는 별도 검증이 필요하다.")

# ---------------- 3) 민감도: 돌발 수요(전시 전환 가정) ----------------
sens = []
for k in (0.8, 1.0, 1.2, 1.4, 1.6):
    f, l = fixed_assign(200 * k, ("A2",)), lp_assign(200 * k, ("A2",))
    sens.append((f"x{k}", 200 * k, max(l["util"].values()), 1 - cost_of(l) / cost_of(f)))
sdf = pd.DataFrame(sens, columns=["수요 배율", "수요", "LP 최대 이용률", "절감률"])
print("[3] 돌발 수요 민감도(A2 폐쇄 상황):", "; ".join(f"{r['수요 배율']} -> 이용률 {r['LP 최대 이용률']:.2f}, 절감 {r['절감률']:.0%}" for _, r in sdf.iterrows()))
print("    수요 x1.2 이상에서는 최대유량(220대/시)을 넘어 우회만으로 수용할 수 없으므로 물량 조정 결정이 필요하다.")

# ---------------- 4) HITL 결재 워크플로우 ----------------
USERS = {"plan_a": "PLN", "plan_b": "PLN", "ops_kim": "OPS", "ops_lee": "OPS", "sup_park": "SUP", "aud_han": "AUD"}
PERM = {"PLN": {"submit", "redraft"}, "OPS": {"approve", "reject"}, "SUP": {"approve", "reject"}, "AUD": {"read"}}


class Flow:
    def __init__(self):
        self.docs, self.log, self.blocked = {}, [], []

    def _audit(self, user, action, doc, detail):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        rec = {"seq": len(self.log) + 1, "user": user, "action": action, "doc": doc, "detail": detail, "prev": prev}
        rec["hash"] = hashlib.sha256(json.dumps(rec, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:12]
        self.log.append(rec)

    def verify(self):
        prev = "GENESIS"
        for r in self.log:
            body = {k: v for k, v in r.items() if k != "hash"}
            if r["prev"] != prev or hashlib.sha256(json.dumps(body, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:12] != r["hash"]:
                return False, r["seq"]
            prev = r["hash"]
        return True, None

    def deny(self, user, act, doc, why):
        self.blocked.append((user, act, doc, why)); self._audit(user, "DENIED:" + act, doc, why); return False

    def submit(self, user, scen, parent=None):
        need = "redraft" if parent else "submit"
        if need not in PERM.get(USERS.get(user), set()):
            return self.deny(user, need, "-", "상신 권한 없음")
        D, cl, cm, l, cf, cl_ = plans[scen]
        extra = l["avg_min"] - plans["평시"][3]["avg_min"]
        big = extra >= 3 or max(l["util"].values()) > 0.98                  # 추가 소요 3분 이상 또는 용량 포화 -> 2단계 결재
        did = f"ORD{len(self.docs) + 1:02d}"
        self.docs[did] = {"scen": scen, "line": ["OPS", "SUP"] if big else ["OPS"], "step": 0, "status": "검토대기", "drafter": user, "done": [], "parent": parent}
        self._audit(user, "submit", did, f"{scen} 추가소요 {extra:+.1f}분")
        return did

    def decide(self, user, did, act, reason=""):
        role, d = USERS.get(user), self.docs.get(did)
        if role is None: return self.deny(user, act, did, "미등록 사용자")
        if d is None or d["status"] != "검토대기": return self.deny(user, act, did, "결재 대상 아님(처리 완료)")
        if act not in PERM[role]: return self.deny(user, act, did, f"{role} 역할은 {act} 불가")
        if d["line"][d["step"]] != role: return self.deny(user, act, did, "결재 순서 위반")
        if user == d["drafter"] or user in d["done"]: return self.deny(user, act, did, "기안자·기결재자 재결재 금지")
        if len(reason.strip()) < 10: return self.deny(user, act, did, "사유 10자 이상 필수")
        if act == "reject":
            d["status"] = "반려"
        else:
            d["done"].append(user); d["step"] += 1
            d["status"] = "승인" if d["step"] == len(d["line"]) else "검토대기"
        self._audit(user, act, did, reason); return True

    def edit(self, user, did):
        if self.docs[did]["status"] == "승인": return self.deny(user, "edit", did, "승인 후 명령 변경 금지(재기안)")
        return True


wf = Flow()
names = ["A2 교량 폐쇄", "A2 교량 정비(용량 65%)", "수송량 폭증(+30%)", "B1 정비 + 폭증", "A2 폐쇄 + 우천", "평시"]
ids = [wf.submit("plan_a" if i % 2 == 0 else "plan_b", n) for i, n in enumerate(names)]
print(f"\n[4] AI 우회안 {len(ids)}건 전부 '검토대기' 등록, 자동 승인 {sum(wf.docs[i]['status'] == '승인' for i in ids)}건 (목표 0건). 2단계 결재 대상 {[i for i in ids if len(wf.docs[i]['line']) == 2]}")
wf.decide("ops_kim", ids[0], "reject", "추가 소요와 용량 포화 우려, 물량 조정안 포함해 재기안")
new = wf.submit("plan_a", names[0], parent=ids[0])
wf.decide("ops_kim", new, "approve", "우회 경로 용량 확인, 상급 승인 상신함")
wf.decide("sup_park", new, "approve", "우회 시행, 운행 중 지체 보고 조건")
wf.decide("ops_lee", ids[1], "approve", "정비 구간 용량 감소분 우회 반영 타당")
wf.decide("ops_kim", ids[5], "approve", "평시 물량으로 변경 사유 없어 기존 유지")
attacks = [("plan_a", ids[2], "approve", "내가 올린 안을 직접 승인한다", "요청자 자기 승인"), ("aud_han", ids[2], "approve", "감사 계정으로 승인 시도", "감사 역할 승인"),
           ("ghost", ids[2], "approve", "외부 계정으로 승인 시도", "미등록 사용자"), ("ops_kim", ids[2], "approve", "", "사유 없는 승인"),
           ("ops_kim", ids[2], "reject", "짧음", "사유 짧은 반려"), ("sup_park", ids[2], "approve", "상급자가 순서를 건너뛰어 승인", "결재 순서 건너뛰기"),
           ("ops_kim", ids[5], "approve", "이미 승인된 문서를 다시 승인", "처리 완료 재승인"), ("sup_park", new, "reject", "승인된 문서를 뒤집어 반려", "승인 후 번복"),
           ("plan_b", ids[3], "approve", "요청자 계정으로 승인 시도함", "요청자 권한 위반")]
for u, doc, act, rs, _ in attacks:
    wf.decide(u, doc, act, rs)
wf.edit("plan_a", new)                                                           # 10) 승인 후 명령 변경
print(f"    권한 우회 시도 10건 중 차단 {len(wf.blocked)}건 (차단율 {len(wf.blocked) / 10:.0%}, 목표 100%)")
for u, a, doc, why in wf.blocked[:4]:
    print(f"      차단 {u:<9} {a:<8} {doc}: {why}")
ok, _ = wf.verify()
snap = json.loads(json.dumps(wf.log))
wf.log[3]["detail"] = "사후에 고쳐 쓴 사유"
ok2, at = wf.verify()
wf.log = snap
print(f"    감사 로그 {len(wf.log)}건 해시 체인 무결성 {'100%' if ok else '훼손'}; 4번 기록 사후 수정 시 {'통과' if ok2 else f'위변조 검출({at}번)'}")
print(f"    {ids[0]}(반려) -> {new}(재기안, parent 연결) 이력:", [(r['action'], r['user']) for r in wf.log if r['doc'] in (ids[0], new)])
print("    문서 상태:", pd.Series({k: v["status"] for k, v in wf.docs.items()}).value_counts().to_dict(), "| 실제 우회 명령 하달은 지휘계통(작전군수관 승인) 결심에 따른다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(res)); ax[0].bar(x - .2, res["고정 비용(만원/시)"], .4, label="fixed share"); ax[0].bar(x + .2, res["LP 비용(만원/시)"], .4, label="LP")
ax[0].set_xticks(x); ax[0].set_xticklabels([f"S{i + 1}" for i in x]); ax[0].legend(); ax[0].set_title("Cost per hour by scenario")
ax[1].plot(sdf["수요"], sdf["절감률"] * 100, "o-"); ax[1].set_title("Sensitivity: demand vs saving (%), A2 closed"); ax[1].set_xlabel("demand (veh/h)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[1] 시나리오별 비용 비교(시간·유류 단가는 교육용 가정값)
            시나리오  수요  고정 배분(분)  LP(분)  고정 비용(만원/시)  LP 비용(만원/시)   절감률
              평시 200      41.9   41.8       1124.8       1122.7  0.2%
    수송량 폭증(+30%) 260      57.2   44.8       1860.5       1547.9 16.8%
A2 교량 정비(용량 65%) 200      53.3   42.2       1351.6       1138.9 15.7%
        A2 교량 폐쇄 200      91.4   45.1       2122.3       1207.2 43.1%
      A2 폐쇄 + 우천 200     119.7   47.0       2688.8       1245.4 53.7%
      B1 정비 + 폭증 250      76.5   46.2       2270.9       1523.7 32.9%
    시나리오 6종 평균 절감 27.1%, 우회가 필요한 4개 시나리오 평균 36.4% (목표 15% 이상: 달성), 평시는 0.2%에 그친다

[2] 백테스팅(30일): 평균 절감 16.0%(목표 15% 이상: 달성), 최대 48.5%(day 134), 절감 0 이하인 날 0일
    절감 10% 미만인 날 12일: 효과는 정비·폭증이 겹친 날에 집중된다. 과거 관행 배분은 가정이므로 실제 부대 실적으로는 별도 검증이 필요하다.
[3] 돌발 수요 민감도(A2 폐쇄 상황): x0.8 -> 이용률 0.85, 절감 4%; x1.0 -> 이용률 0.95, 절감 43%; x1.2 -> 이용률 1.32, 절감 58%; x1.4 -> 이용률 1.54, 절감 58%; x1.6 -> 이용률 1.76, 절감 56%
    수요 x1.2 이상에서는 최대유량(220대/시)을 넘어 우회만으로 수용할 수 없으므로 물량 조정 결정이 필요하다.

[4] AI 우회안 6건 전부 '검토대기' 등록, 자동 승인 0건 (목표 0건). 2단계 결재 대상 ['ORD01', 'ORD03', 'ORD04', 'ORD05']
    권한 우회 시도 10건 중 차단 10건 (차단율 100%, 목표 100%)
      차단 plan_a    approve  ORD03: PLN 역할은 approve 불가
      차단 aud_han   approve  ORD03: AUD 역할은 approve 불가
      차단 ghost     approve  ORD03: 미등록 사용자
      차단 ops_kim   approve  ORD03: 사유 10자 이상 필수
    감사 로그 22건 해시 체인 무결성 100%; 4번 기록 사후 수정 시 위변조 검출(4번)
    ORD01(반려) -> ORD07(재기안, parent 연결) 이력: [('submit', 'plan_a'), ('reject', 'ops_kim'), ('submit', 'plan_a'), ('approve', 'ops_kim'), ('approve', 'sup_park'), ('DENIED:reject', 'sup_park'), ('DENIED:edit', 'plan_a')]
    문서 상태: {'승인': 3, '검토대기': 3, '반려': 1} | 실제 우회 명령 하달은 지휘계통(작전군수관 승인) 결심에 따른다.
```
</details>
